# Train Reader trên Colab — B1 (xlm-roberta-base trích xuất)

1. Runtime → Change runtime type → **T4 GPU**
2. Trên máy local tạo zip (kèm file `COMMIT` để mỗi file dự đoán ghi được phiên bản code):
   `git archive --add-virtual-file=COMMIT:$(git rev-parse --short HEAD) -o hmtt.zip HEAD src configs data/processed notebooks requirements.txt`
   rồi đặt vào Drive: `MyDrive/HMTT/hmtt.zip`
3. Runtime → **Run all**. Không cần restart: mọi bước chạy bằng `!python` (tiến trình mới) nên luôn dùng đúng bản `transformers==4.57.6` vừa cài.

Kết quả tự lưu vào `MyDrive/HMTT/predictions/` và `MyDrive/HMTT/models/`. Mất khoảng 30–40 phút trên T4.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/HMTT'
!rm -rf /content/HMTT && mkdir -p /content/HMTT && cd /content/HMTT && unzip -q $DRIVE/hmtt.zip
%cd /content/HMTT
import subprocess
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv'], capture_output=True, text=True)
print(gpu.stdout or gpu.stderr)
# Lệnh `!` lỗi KHÔNG dừng Run all -> kiểm GPU bằng assert Python để dừng hẳn
assert gpu.returncode == 0, 'KHÔNG có GPU: Runtime -> Change runtime type -> T4 GPU, rồi Run all lại'

In [ ]:
# Colab đã có torch khớp CUDA -> chỉ cài phần còn lại theo bản ghim
!pip install -q -r requirements.txt
!python -c "import transformers, torch; print(transformers.__version__, torch.__version__, torch.cuda.is_available()); assert transformers.__version__ == '4.57.6'; assert torch.cuda.is_available(), 'Chưa bật GPU'"

In [ ]:
# B1: train trên train (đoạn đúng), chọn ngưỡng null trên dev, dự đoán dev
!PYTHONIOENCODING=utf-8 python src/reader/train_extractive.py --config configs/reader_extractive.yaml --tune_threshold 2>&1 | grep -v Warning
!PYTHONIOENCODING=utf-8 python src/eval/evaluate_pipeline.py data/predictions/b1_gold_dev.jsonl

In [ ]:
# Lưu lên Drive (checkpoint KHÔNG đưa vào git)
!mkdir -p $DRIVE/predictions $DRIVE/models
!cp data/predictions/*.jsonl $DRIVE/predictions/
!rm -rf $DRIVE/models/b1_xlmr && cp -r models/b1_xlmr $DRIVE/models/
!ls -la $DRIVE/predictions